In [25]:
import os, requests
from pypdf import PdfReader
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool


In [26]:
if "GEMINI_API_KEY" not in os.environ: raise RuntimeError("Set GEMINI_API_KEY in the environment before running this cell")
ADZUNA_ID =  "ea56c5f4"
ADZUNA_KEY = os.environ["ADZUNA_KEY"]

llm = LLM(model="gemini/gemini-3.5-flash")

In [27]:
# read resume
pages = PdfReader("Milan_Resume.pdf").pages
resume_text = "\n".join(p.extract_text() for p in pages)

# # tool: agent can call this to search jobs
# @tool("Job Search")
# def job_search(query: str, location: str = "Kochi") -> str:
#     """Search current job listings. Input: job title keywords and a city."""
#     r = requests.get(
#         "https://api.adzuna.com/v1/api/jobs/in/search/1",
#         params={"app_id": ADZUNA_ID, "app_key": ADZUNA_KEY,
#                 "what": query, "where": location,
#                 "results_per_page": 10},
#         timeout=30,
#     )
#     jobs = r.json().get("results", [])
#     if not jobs:
#         return "No jobs found. Try other keywords or city."
#     return "\n".join(
#         f"{j['title']} | {j['company']['display_name']} | "
#         f"{j['location']['display_name']} | {j['redirect_url']}"
#         for j in jobs
#     )

In [28]:
@tool("Job Search")
def job_search(query: str, location: str = "India") -> str:
    """
    Search Adzuna for real job listings.

    Input:
    - query: job keywords such as Python, AI Engineer, Data Scientist
    - location: city or country such as Kochi, Bangalore, India, Remote
    """

    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

    params = {
        "app_id": ADZUNA_ID,
        "app_key": ADZUNA_KEY,
        "what": query,
        "where": location,
        "results_per_page": 20,
        "content-type": "application/json"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=30
        )

        # Important for debugging
        print("\n================ ADZUNA SEARCH ================")
        print("Query:", query)
        print("Location:", location)
        print("Status:", response.status_code)

        if response.status_code != 200:
            print("Adzuna response:", response.text)
            return f"Adzuna API error: {response.status_code} - {response.text}"

        data = response.json()

        jobs = data.get("results", [])

        print("Jobs found:", len(jobs))
        print("================================================\n")

        if not jobs:
            return (
                f"No jobs found for query='{query}' "
                f"and location='{location}'."
            )

        results = []

        for job in jobs:

            title = job.get("title", "Unknown title")

            company = job.get(
                "company",
                {}
            ).get(
                "display_name",
                "Unknown company"
            )

            job_location = job.get(
                "location",
                {}
            ).get(
                "display_name",
                location
            )

            description = job.get(
                "description",
                ""
            )

            url = job.get(
                "redirect_url",
                ""
            )

            salary_min = job.get("salary_min")
            salary_max = job.get("salary_max")

            salary = ""

            if salary_min and salary_max:
                salary = f"Salary: {salary_min} - {salary_max}"
            elif salary_min:
                salary = f"Salary from: {salary_min}"

            results.append(
                f"""
TITLE: {title}
COMPANY: {company}
LOCATION: {job_location}
{salary}
DESCRIPTION: {description}
URL: {url}
"""
            )

        return "\n-----------------------------\n".join(results)

    except requests.exceptions.RequestException as e:
        return f"Request failed: {str(e)}"

    except Exception as e:
        return f"Unexpected error: {str(e)}"

In [29]:
analyzer = Agent(
    role="Resume Analyzer",
    goal="Find skills, experience level, and best job titles from a resume",
    backstory="You read resumes carefully and never invent skills.",
    llm=llm, verbose=True,
)

hunter = Agent(
    role="Job Hunter",
    goal="Find real job openings that match the candidate",
    backstory="You search many keywords and cities, and only report jobs the tool returned.",
    tools=[job_search],
    llm=llm, verbose=True,
)

analyze_task = Task(
    description=("Read this resume:\n{resume}\n\n"
                 "List: skills, experience level, and 5 job titles that fit. "
                 "Candidate is aiming for AI/ML engineer and data scientist roles."),
    expected_output="Skills list, experience level, 5 job titles",
    agent=analyzer,
)

hunt_task = Task(
    description=("Using the job titles from the analysis, use the Job Search tool. "
                 "Search each title in Kochi, Bangalore, and Remote. "
                 "Pick the 10 best matches for a fresher or early-career candidate."),
    expected_output=("Table of 10 jobs: title, company, location, link, "
                     "and one line on why it matches. Only use jobs the tool returned."),
    agent=hunter,
    output_file="jobs.md",
)

crew = Crew(agents=[analyzer, hunter], tasks=[analyze_task, hunt_task], verbose=True)
result = crew.kickoff(inputs={"resume": resume_text})
print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.1                                                                                        │
│  Latest version:  1.15.23                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b3aa92f1-1393-43ca-b68c-1f6a294ad7ac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Read this resume:                                                                                        │
│  MILAN K ANTONY                                                                                                 │
│  Ernakulam,Kerala-682027                                                                                        │
│  milanantony005@gmail.com                                                                                       │
│  OBJECTIVE                                                                                                      │
│  Instrumentation and Control Engineering student with a strong foundation in sensor technology and control      │
│  systems, looking for an job opportunity in a fast-paced industrial environment. Driven to explore the          │
│  integration                                                                                                    │
│  of advanced automation and real-time data monitoring to improve operational reliability. Committed to          │
│  applying                                                                                                       │
│  a proactive learning mindset to solve technical challenges within innovative engineering teams.                │
│  COURSE OUTCOME                                                                                                 │
│  Highly technical Instrumentation and Control student with a focused background in process automation and in-   │
│  dustrial control logic. Proficient in developing PID tuning strategies and configuring PLC, SCADA, and DCS     │
│  environments. Skilled in the end-to-end management of smart field instrumentation—from selection and calibra-  │
│  tion to maintenance—with a professional-grade ability to interpret and execute complex P”&”IDs.                │
│  EDUCA TION                                                                                                     │
│  Bachelor of T echnology in Instrumentation and Control Engineering,ErnakulamExpected 2027                      │
│  Cochin University of Science and Technology CGPA : 6.506/5                                                     │
│  Higher secondary ,Class 12(Kerala State)2021 - 2023                                                            │
│  HSS of Jesus, Kothad, Ernakulam, 83.91%                                                                        │
│  High School,SSLC 2021                                                                                          │
│  HIBHS, Varapuzha, Ernakulam, Above 95%                                                                         │
│  SKILLS                                                                                                         │
│  Soft SkillsTeamwork and Collaboration, Adaptability, Quick Learning, Logical thinking                          │
│  LanguagesEnglish, Malayalam, Hindi, Tamil                                                                      │
│  TECHNICAL KNOWLEDGE                                                                                            │
│  •CProgramming                                                                                                  │
│  •MATLAB                                                                                                        │
│  •Simulink                                                                                                      │
│  •Labview                                                                                                       │
│  •LTspice                                              

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resume Analyzer                                                                                         │
│                                                                                                                 │
│  Task: Read this resume:                                                                                        │
│  MILAN K ANTONY                                                                                                 │
│  Ernakulam,Kerala-682027                                                                                        │
│  milanantony005@gmail.com                                                                                       │
│  OBJECTIVE                                                                                                      │
│  Instrumentation and Control Engineering student with a strong foundation in sensor technology and control      │
│  systems, looking for an job opportunity in a fast-paced industrial environment. Driven to explore the          │
│  integration                                                                                                    │
│  of advanced automation and real-time data monitoring to improve operational reliability. Committed to          │
│  applying                                                                                                       │
│  a proactive learning mindset to solve technical challenges within innovative engineering teams.                │
│  COURSE OUTCOME                                                                                                 │
│  Highly technical Instrumentation and Control student with a focused background in process automation and in-   │
│  dustrial control logic. Proficient in developing PID tuning strategies and configuring PLC, SCADA, and DCS     │
│  environments. Skilled in the end-to-end management of smart field instrumentation—from selection and calibra-  │
│  tion to maintenance—with a professional-grade ability to interpret and execute complex P”&”IDs.                │
│  EDUCA TION                                                                                                     │
│  Bachelor of T echnology in Instrumentation and Control Engineering,ErnakulamExpected 2027                      │
│  Cochin University of Science and Technology CGPA : 6.506/5                                                     │
│  Higher secondary ,Class 12(Kerala State)2021 - 2023                                                            │
│  HSS of Jesus, Kothad, Ernakulam, 83.91%                                                                        │
│  High School,SSLC 2021                                                                                          │
│  HIBHS, Varapuzha, Ernakulam, Above 95%                                                                         │
│  SKILLS                                                                                                         │
│  Soft SkillsTeamwork and Collaboration, Adaptability, Quick Learning, Logical thinking                          │
│  LanguagesEnglish, Malayalam, Hindi, Tamil                                                                      │
│  TECHNICAL KNOWLEDGE                                                                                            │
│  •CProgramming                                                                                                  │
│  •MATLAB                                                                                                        │
│  •Simulink                                                                                                      │
│  •Labview                                              

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Job Hunter                                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Job Title | Company | Location | Application Link | Why It Matches |                                         │
│  | :--- | :--- | :--- | :--- | :--- |                                                                           │
│  | **Control System Engineer** | Flux Auto | Bangalore, Karnataka | [Apply                                      │
│  Here](https://www.adzuna.in/details/2836410644?utm_medium=api&utm_source=ea56c5f4) | Focuses on developing     │
│  autonomous vehicle control hardware and software, aligning perfectly with control systems, sensor technology,  │
│  and vehicle automation. |                                                                                      │
│  | **Junior Computer Vision & Robotics Engineer** | SkyMul | Kochi, Ernakulam | [Apply                          │
│  Here](https://www.adzuna.in/land/ad/5864427628?se=oNFLPja88RGJArKjYfGXsA&utm_medium=api&utm_source=ea56c5f4&v  │
│  =FE1235DC0098B95416D26F0D7838C36E49B96A10) | Located in Kochi, this entry-level role involves ROS2, sensor     │
│  fusion, and hardware-to-actuation pipelines, utilizing the candidate's sensor and programming skills. |        │
│  | **Matlab Developer** | Bullhorn Consultants | Bangalore, Karnataka | [Apply                                  │
│  Here](https://www.adzuna.in/details/1286247986?utm_medium=api&utm_source=ea56c5f4) | Directly aligns with the  │
│  candidate's core technical proficiency in MATLAB and Simulink development. |                                   │
│  | **Matlab / Simscape Developer** | Robosoft Technologies | Bangalore, Karnataka | [Apply                      │
│  Here](https://www.adzuna.in/details/5859122673?utm_medium=api&utm_source=ea56c5f4) | Leverages the             │
│  candidate's mathematical modeling and MATLAB/Simulink academic background for physical system simulation. |    │
│  | **MATLAB Specialist - Scientific Computing** | Mercor | Remote / Bangalore | [Apply                          │
│  Here](https://www.adzuna.in/details/5883377940?utm_medium=api&utm_source=ea56c5f4) | A remote-friendly role    │
│  centering entirely on numerical computing, simulation, and signal processing using MATLAB. |                   │
│  | **Data Analyst** | ALTRAIZE | Bangalore, Karnataka | [Apply                                                  │
│  Here](https://www.adzuna.in/details/5899028729?utm_medium=api&utm_source=ea56c5f4) | Focuses on analyzing      │
│  manufacturing, production, quality, and maintenance data, serving as a perfect process/industrial data         │
│  analyst role. |                                                                                                │
│  | **Data Analyst Intern** | Futurlytic Private Limited | Bangalore, Karnataka | [Apply                         │
│  Here](https://www.adzuna.in/land/ad/5901560250?se=hk1OPja88RG5QYXapx_3iw&utm_medium=api&utm_source=ea56c5f4&v  │
│  =DE2F292EE84B55235A190DB69867F0C2DA8B10C2) | Targeted at technical students/interns, helping the candidate     │
│  bridge into data analytics before graduating in 2027. |                                                        │
│  | **Embedded Software Engineer - Sensor I/O** | KG Information Systems | Bangalore, Karnataka | [Apply         │
│  Here](https://www.adzuna.in/details/5859122308?utm_medium=api&utm_source=ea56c5f4) | Combines C programming,   │
│  sensor integration, sensor fusion, and MATLAB, fitting

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the job titles from the analysis, use the Job Search tool. Search each title in Kochi, Bangalore,  │
│  and Remote. Pick the 10 best matches for a fresher or early-career candidate.                                  │
│  Agent: Job Hunter                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: b3aa92f1-1393-43ca-b68c-1f6a294ad7ac                                                                       │
│  Final Output: | Job Title | Company | Location | Application Link | Why It Matches |                           │
│  | :--- | :--- | :--- | :--- | :--- |                                                                           │
│  | **Control System Engineer** | Flux Auto | Bangalore, Karnataka | [Apply                                      │
│  Here](https://www.adzuna.in/details/2836410644?utm_medium=api&utm_source=ea56c5f4) | Focuses on developing     │
│  autonomous vehicle control hardware and software, aligning perfectly with control systems, sensor technology,  │
│  and vehicle automation. |                                                                                      │
│  | **Junior Computer Vision & Robotics Engineer** | SkyMul | Kochi, Ernakulam | [Apply                          │
│  Here](https://www.adzuna.in/land/ad/5864427628?se=oNFLPja88RGJArKjYfGXsA&utm_medium=api&utm_source=ea56c5f4&v  │
│  =FE1235DC0098B95416D26F0D7838C36E49B96A10) | Located in Kochi, this entry-level role involves ROS2, sensor     │
│  fusion, and hardware-to-actuation pipelines, utilizing the candidate's sensor and programming skills. |        │
│  | **Matlab Developer** | Bullhorn Consultants | Bangalore, Karnataka | [Apply                                  │
│  Here](https://www.adzuna.in/details/1286247986?utm_medium=api&utm_source=ea56c5f4) | Directly aligns with the  │
│  candidate's core technical proficiency in MATLAB and Simulink development. |                                   │
│  | **Matlab / Simscape Developer** | Robosoft Technologies | Bangalore, Karnataka | [Apply                      │
│  Here](https://www.adzuna.in/details/5859122673?utm_medium=api&utm_source=ea56c5f4) | Leverages the             │
│  candidate's mathematical modeling and MATLAB/Simulink academic background for physical system simulation. |    │
│  | **MATLAB Specialist - Scientific Computing** | Mercor | Remote / Bangalore | [Apply                          │
│  Here](https://www.adzuna.in/details/5883377940?utm_medium=api&utm_source=ea56c5f4) | A remote-friendly role    │
│  centering entirely on numerical computing, simulation, and signal processing using MATLAB. |                   │
│  | **Data Analyst** | ALTRAIZE | Bangalore, Karnataka | [Apply                                                  │
│  Here](https://www.adzuna.in/details/5899028729?utm_medium=api&utm_source=ea56c5f4) | Focuses on analyzing      │
│  manufacturing, production, quality, and maintenance data, serving as a perfect process/industrial data         │
│  analyst role. |                                                                                                │
│  | **Data Analyst Intern** | Futurlytic Private Limited | Bangalore, Karnataka | [Apply                         │
│  Here](https://www.adzuna.in/land/ad/5901560250?se=hk1OPja88RG5QYXapx_3iw&utm_medium=api&utm_source=ea56c5f4&v  │
│  =DE2F292EE84B55235A190DB69867F0C2DA8B10C2) | Targeted at technical students/interns, helping the candidate     │
│  bridge into data analytics before graduating in 2027. |                                                        │
│  | **Embedded Software Engineer - Sensor I/O** | KG Information Systems | Bangalore, Karnataka | [Apply         │
│  Here](https://www.adzuna.in/details/5859122308?utm_medium=api&utm_source=ea56c5f4) | Combines C programming,   │
│  sensor integration, sensor fusion, and MATLAB, fittin

| Job Title | Company | Location | Application Link | Why It Matches |
| :--- | :--- | :--- | :--- | :--- |
| **Control System Engineer** | Flux Auto | Bangalore, Karnataka | [Apply Here](https://www.adzuna.in/details/2836410644?utm_medium=api&utm_source=ea56c5f4) | Focuses on developing autonomous vehicle control hardware and software, aligning perfectly with control systems, sensor technology, and vehicle automation. |
| **Junior Computer Vision & Robotics Engineer** | SkyMul | Kochi, Ernakulam | [Apply Here](https://www.adzuna.in/land/ad/5864427628?se=oNFLPja88RGJArKjYfGXsA&utm_medium=api&utm_source=ea56c5f4&v=FE1235DC0098B95416D26F0D7838C36E49B96A10) | Located in Kochi, this entry-level role involves ROS2, sensor fusion, and hardware-to-actuation pipelines, utilizing the candidate's sensor and programming skills. |
| **Matlab Developer** | Bullhorn Consultants | Bangalore, Karnataka | [Apply Here](https://www.adzuna.in/details/1286247986?utm_medium=api&utm_source=ea56c5f4) | Direct

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯